# 05 UC3M-LP check (M1b-6)
Runs on Colab. UC3M-LP (Spanish plates, Zenodo 17152029) is an **EU test set only**: never train on it, never share converted copies. Plan: PLAN.md (M1b-6).

Inspect first, convert later: this notebook downloads the zip, lists what is inside, and counts odd characters in its text files. The conversion cells follow Cell C, written after the real label format was seen (one JSON per image, plate polygons, authors' train/test folders). Run the cells top to bottom. Each cell prints its elapsed seconds.
Cell A prints folder and file names, so it is tagged `remove_output`; its output is not saved in the committed notebook.

In [5]:
import subprocess, time
t0 = time.perf_counter()
SCRIPT = '''
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition
'''
subprocess.run(["bash", "-c", SCRIPT], check=True)
print(f"elapsed {time.perf_counter() - t0:.1f} s")

elapsed 2.6 s


In [6]:
import csv
import io
import json
import pprint
import random
import tarfile
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from google.colab import drive
from alpr import uc3m

t0 = time.perf_counter()
# Parameters
LOCAL = Path("/content/UC3M-LP.zip")
DRIVE_ZIP = Path("/content/drive/MyDrive/alpr/archives/UC3M-LP.zip")
README_MAX = 20 * 1024   # README/.md/.txt text is printed only for files up to this many bytes
README_FILES = 5         # at most this many doc files are looked at
TEXT_EXT = (".txt", ".json", ".xml", ".csv", ".yaml", ".yml")  # files scanned by the odd-character count
TEXT_MAX = 1 << 20       # text files above this many bytes are skipped (and counted)
OUT_TAR = Path("/content/uc3m_yolo.tar")  # built on the VM disk, then copied to Drive with its two CSVs
DRIVE_DIR = Path("/content/drive/MyDrive/alpr/archives")
SEED = 0
N_CHECK = 12             # random images in the visual box check

drive.mount("/content/drive")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
elapsed 0.5 s


In [22]:
t0 = time.perf_counter()
!python -m alpr.uc3m download {LOCAL} {DRIVE_ZIP}
print(f"elapsed {time.perf_counter() - t0:.1f} s")

/content/drive/MyDrive/alpr/archives/UC3M-LP.zip: MD5 ok, skipping
elapsed 20.5 s


In [ ]:
# Cell A: zip listing (prints names, tagged remove_output). Opens the zip read-only, extracts nothing.
t0 = time.perf_counter()
ZIP = LOCAL if LOCAL.exists() else DRIVE_ZIP
with zipfile.ZipFile(ZIP) as z:
    infos = [i for i in z.infolist() if not i.is_dir()]
    print(f"{ZIP}: {len(infos)} files, {sum(i.file_size for i in infos) / 1e9:.2f} GB uncompressed")
    folder = lambda n: "/".join(PurePosixPath(n).parts[:-1][:2]) or "(root)"
    ext = lambda n: PurePosixPath(n).suffix.lower() or "(none)"
    print("files per folder (first two levels):", dict(sorted(Counter(folder(i.filename) for i in infos).items())))
    print("files per extension:", dict(sorted(Counter(ext(i.filename) for i in infos).items())))
    print("files per (folder, extension):", dict(sorted(Counter((folder(i.filename), ext(i.filename)) for i in infos).items())))

    def is_doc(name: str) -> bool:
        base = name.lower().rsplit("/", 1)[-1]
        return base.startswith("readme") or base.endswith(".md") or (base.endswith(".txt") and name.count("/") <= 1)

    docs = [i for i in infos if is_doc(i.filename)]
    print(f"\ndoc-like files: {len(docs)} (looking at {min(len(docs), README_FILES)})")
    for i in docs[:README_FILES]:
        if i.file_size <= README_MAX:
            print(f"--- {i.filename} ({i.file_size} bytes)")
            print(z.read(i).decode("utf-8", "replace"))
        else:
            print(f"--- {i.filename}: {i.file_size} bytes, over {README_MAX}, not printed")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

In [24]:
# Blurred-character count: provisional until the label format is known (which symbol marks a blurred character is a guess).
# Counts every character that is not whitespace, a digit or A-Z, over the text-like members. No plate text, no names are printed.
t0 = time.perf_counter()
ZIP = LOCAL if LOCAL.exists() else DRIVE_ZIP
chars, files_with = Counter(), Counter()
with zipfile.ZipFile(ZIP) as z:
    members = [i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith(TEXT_EXT)]
    texts = [i for i in members if i.file_size <= TEXT_MAX]
    for i in texts:
        odd = Counter(c for c in z.read(i).decode("utf-8", "replace") if not (c.isspace() or "0" <= c <= "9" or "A" <= c <= "Z"))
        chars.update(odd)
        files_with.update(odd.keys())
print(f"text-like members: {len(members)}, scanned: {len(texts)}, skipped as too big: {len(members) - len(texts)}")
for c, n in chars.most_common(30):
    print(f"{c!r}: {n} times in {files_with[c]} files")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

text-like members: 1977, scanned: 1977, skipped as too big: 0
'"': 116668 times in 1975 files
',': 91016 times in 1975 files
'[': 55707 times in 1975 files
']': 55707 times in 1975 files
'o': 45912 times in 1975 files
':': 41055 times in 1975 files
'c': 33155 times in 1975 files
'r': 33155 times in 1975 files
'd': 32583 times in 1975 files
'_': 30608 times in 1975 files
'a': 25751 times in 1975 files
'b': 25514 times in 1975 files
'i': 25179 times in 1975 files
'h': 21229 times in 1975 files
'{': 17279 times in 1975 files
'}': 17279 times in 1975 files
'x': 12757 times in 1975 files
'e': 10447 times in 1975 files
'g': 9875 times in 1975 files
'p': 9044 times in 1975 files
't': 8472 times in 1975 files
'l': 7069 times in 1975 files
'm': 5925 times in 1975 files
'*': 5094 times in 1975 files
's': 4522 times in 1975 files
'-': 2547 times in 1975 files
'y': 2547 times in 1975 files
'.': 1975 times in 1975 files
'j': 1975 times in 1975 files
elapsed 1.1 s


In [ ]:
# Cell C: JSON structure of the labels (key paths, patterns and counts only; tagged remove_output). Reads the zip, extracts nothing.
t0 = time.perf_counter()
ZIP = LOCAL if LOCAL.exists() else DRIVE_ZIP
pat = lambda s: "".join("A" if c.isalpha() else "9" if c.isdigit() else c for c in s)  # letters -> A, digits -> 9
safe_key = lambda k: k if k.replace("_", "").isalpha() and len(k) <= 30 else pat(k)    # odd keys are shown as patterns too
tname = lambda v: "list" if isinstance(v, list) else "dict" if isinstance(v, dict) else "string" if isinstance(v, str) else "bool" if isinstance(v, bool) else "null" if v is None else "number"
shape = lambda v: ({safe_key(k): shape(x) for k, x in v.items()} if isinstance(v, dict) else [shape(x) for x in v] if isinstance(v, list) else pat(v) if isinstance(v, str) else v)
occ, nfiles, lens = defaultdict(Counter), Counter(), defaultdict(Counter)
allnum, lo, hi, stars, state = {}, {}, {}, Counter(), {"star_files": 0}

def walk(v, path, seen, fstate):
    t = tname(v)
    occ[path][t] += 1
    seen.add(path)
    if t == "dict":
        for k, x in v.items():
            walk(x, f"{path}.{safe_key(k)}", seen, fstate)
    elif t == "list":
        lens[path][len(v)] += 1
        allnum[path] = allnum.get(path, True) and all(tname(x) == "number" for x in v)
        for x in v:
            walk(x, path + "[]", seen, fstate)
    elif t == "number":
        lo[path], hi[path] = min(lo.get(path, v), v), max(hi.get(path, v), v)
    elif t == "string" and "*" in v:
        stars[pat(v)] += 1
        fstate["star"] = True

with zipfile.ZipFile(ZIP) as z:
    infos = {i.filename: i for i in z.infolist() if not i.is_dir()}
    docs, bad = {}, 0
    for n in sorted(infos):
        if n.lower().endswith(".json"):
            try:
                docs[n] = json.loads(z.read(n))
            except ValueError:
                bad += 1
    for v in docs.values():
        seen, fstate = set(), {"star": False}
        walk(v, "$", seen, fstate)
        nfiles.update(seen)
        state["star_files"] += fstate["star"]

    print(f"1. JSON files parsed: {len(docs)}, unparseable: {bad}")
    rows = sorted(occ)
    for p in rows[:80]:
        print(f"  {p}: in {nfiles[p]} files, {sum(occ[p].values())} times, types {dict(occ[p])}")
    print(f"  ({len(rows)} key paths in total{', first 80 shown' if len(rows) > 80 else ''})")

    stem = lambda n: PurePosixPath(n).stem
    split_of = lambda n: PurePosixPath(n).parent.name
    natural = lambda s: (not s.isdigit(), int(s) if s.isdigit() else 0, s)
    stems = {sp: sorted((stem(n) for n in docs if split_of(n) == sp), key=natural) for sp in ("train", "test")}
    picks = [("train", stems["train"][0]), ("test", stems["test"][0]), ("train", stems["train"][1])]
    names = {(split_of(n), stem(n)): n for n in docs}
    print("\n2. Sample structures (strings as patterns, numbers kept) and image sizes read from the jpg headers")
    for sp, sid in picks:
        n = names[(sp, sid)]
        print(f"--- {sp}, id #{stems[sp].index(sid) + 1} by sort order")
        print(pprint.pformat(shape(docs[n]), width=100, sort_dicts=False)[:4000])
        jpg = n[:-5] + ".jpg"
        if jpg in infos:
            with Image.open(io.BytesIO(z.read(jpg))) as im:
                print(f"image size (width x height): {im.size[0]} x {im.size[1]}")
        else:
            print("no matching .jpg with the same path")

    print(f"\n3. Strings containing '*': {sum(stars.values())} strings in {state['star_files']} files; top 20 patterns")
    for p, c in stars.most_common(20):
        print(f"  {p!r}: {c}")

    print("\n4. Lengths of every list in the structure (length: how many lists have it)")
    for p in sorted(lens)[:30]:
        print(f"  {p}: {dict(sorted(lens[p].items())[:15])}")

    print("\n5. Numbers: min / max per path (compare with image width x height above)")
    for p in sorted(lo)[:60]:
        print(f"  {p}: min {lo[p]}, max {hi[p]}, {occ[p]['number']} values")
    print("   lists holding only numbers (box or polygon candidates):", {p: dict(lens[p]) for p in lens if allnum.get(p) and set(lens[p]) != {0}})

    print("\n6. train.txt / test.txt ids vs the folders")
    jpgs = [n for n in infos if n.lower().endswith(".jpg")]
    ids = {}
    for sp in ("train", "test"):
        f = [n for n in infos if PurePosixPath(n).name == f"{sp}.txt"]
        lines = [l.strip() for l in z.read(f[0]).decode("utf-8", "replace").splitlines() if l.strip()] if len(f) == 1 else []
        ids[sp] = {PurePosixPath(l).stem for l in lines}
        js, jp = {stem(n) for n in docs if split_of(n) == sp}, {stem(n) for n in jpgs if split_of(n) == sp}
        print(f"  {sp}.txt: {len(f)} file, {len(lines)} lines ({len(ids[sp])} distinct ids), first line pattern {pat(lines[0]) if lines else None!r}")
        print(f"    folder has {len(jp)} jpg and {len(js)} json; ids missing from jpg folder {len(ids[sp] - jp)}, jpgs not in ids {len(jp - ids[sp])}; ids missing from json {len(ids[sp] - js)}, jsons not in ids {len(js - ids[sp])}")
    print(f"  ids in both train.txt and test.txt: {len(ids['train'] & ids['test'])}")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

In [26]:
t0 = time.perf_counter()
ZIP = LOCAL if LOCAL.exists() else DRIVE_ZIP
# Streams the zip (nothing extracted). Stops without writing a tar if any image has an EXIF orientation other than 1: then tell Claude.
!python -m alpr.uc3m yolo {ZIP} {OUT_TAR}
print(f"elapsed {time.perf_counter() - t0:.1f} s")

images found: 1975 | kept: 1975 | dropped: 0
kept images per split: {'test': 395, 'train': 1580} | kept plates per split: {'test': 491, 'train': 2056}
dropped per (reason, split): {}
images with EXIF orientation other than 1: 0 (any would have stopped the run)
elapsed 433.1 s


In [27]:
t0 = time.perf_counter()
# Stops if Drive has under 10 GB free. Copies the tar and the two CSVs via .part and checks the MD5 of every copy.
!python -m alpr.uc3m copy {OUT_TAR} {DRIVE_DIR}
print(f"elapsed {time.perf_counter() - t0:.1f} s")

uc3m_yolo.tar: copied, MD5 of the copy ok (e3ef306731f33095bac983338c6adb04)
uc3m_labels.csv: copied, MD5 of the copy ok (deccc9443b71c73fe247666d8466527a)
uc3m_yolo.dropped.csv: copied, MD5 of the copy ok (ee708d592da4e47ebd302ae7e0dd221b)
elapsed 89.3 s


In [7]:
# Counts-only report on the converted set (no plate text and no file names are printed)
t0 = time.perf_counter()
rows = list(csv.DictReader(open(OUT_TAR.with_name("uc3m_labels.csv"), newline="", encoding="utf-8")))
dropped = list(csv.DictReader(open(OUT_TAR.with_suffix(".dropped.csv"), newline="", encoding="utf-8")))
imgs = {(r["image_id"], r["split"]): (int(r["image_w"]), int(r["image_h"])) for r in rows}
stat = lambda a: f"min {min(a):g}, median {float(np.median(a)):g}, max {max(a):g}"
print("kept images per split:", dict(Counter(s for _, s in imgs)), "| kept plates per split:", dict(Counter(r["split"] for r in rows)))
print("dropped per (reason, split):", dict(Counter((r["reason"], r["split"]) for r in dropped)) or "none")
ws, hs = [w for w, _ in imgs.values()], [h for _, h in imgs.values()]
print(f"image width px: {stat(ws)}; height px: {stat(hs)}")
print(f"landscape {sum(w > h for w, h in imgs.values())}, portrait {sum(h > w for w, h in imgs.values())}, square {sum(h == w for w, h in imgs.values())}")
area = [float(r["w"]) * float(r["h"]) for r in rows]
print(f"box area share: {stat(area)}; under 0.0005: {sum(a < 0.0005 for a in area)}, over 0.5: {sum(a > 0.5 for a in area)}")
lp = [uc3m.split_lp(r["lp_id"]) for r in rows]
print("'*' per plate:", dict(sorted(Counter(s for _, _, s in lp).items())))
print("positions after the prefix:", dict(sorted(Counter(n for _, n, _ in lp).items())))
print("visible characters per plate:", dict(sorted(Counter(int(r["n_visible_chars"]) for r in rows).items())))
prefixes = Counter(p for p, _, _ in lp)
print(f"distinct prefixes: {len(prefixes)}" + ("" if len(prefixes) <= 12 else " (more than 12, not listed)"))
if len(prefixes) <= 12:
    print(f"{'prefix':>8} {'plates':>7} {'7 pos':>6} {'8 pos':>6} {'other':>6} {'train':>6} {'test':>6}")
    for pre, n in prefixes.most_common():
        pos = Counter(k for (q, k, _) in lp if q == pre)
        side = Counter(r["split"] for (q, _, _), r in zip(lp, rows) if q == pre)
        print(f"{pre!r:>8} {n:>7} {pos[7]:>6} {pos[8]:>6} {n - pos[7] - pos[8]:>6} {side['train']:>6} {side['test']:>6}")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

kept images per split: {'test': 395, 'train': 1580} | kept plates per split: {'test': 491, 'train': 2056}
dropped per (reason, split): none
image width px: min 1536, median 6944, max 9248; height px: min 1536, median 4640, max 9248
landscape 1918, portrait 57, square 0
box area share: min 8.61866e-05, median 0.0033183, max 0.240922; under 0.0005: 209, over 0.5: 0
'*' per plate: {2: 2547}
positions after the prefix: {7: 2525, 8: 22}
visible characters per plate: {5: 2525, 6: 22}
distinct prefixes: 9
  prefix  plates  7 pos  8 pos  other  train   test
    'AD'    2155   2144     11      0   1731    424
    'AN'     343    339      4      0    287     56
    'BD'      18     18      0      0     11      7
    'BN'      13     13      0      0     12      1
    'DD'       8      8      0      0      7      1
    'EN'       3      0      3      0      2      1
    'DN'       3      3      0      0      2      1
    'ED'       3      0      3      0      3      0
    'CD'       1      0     

In [ ]:
# visual box check: UC3M-LP (image cell, tagged remove_output). Titles show no ids or names.
t0 = time.perf_counter()
by_img = {}
for r in rows:
    by_img.setdefault((r["image_id"], r["split"]), []).append(r)
picks = random.Random(SEED).sample(sorted(by_img), min(N_CHECK, len(by_img)))
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
with tarfile.open(OUT_TAR) as t:
    for ax, (iid, sp) in zip(axes.flat, picks):
        im = Image.open(io.BytesIO(t.extractfile(f"images/{sp}/{iid}.jpg").read()))
        im.thumbnail((800, 800))
        W, H = im.size
        ax.imshow(im)
        for r in by_img[(iid, sp)]:
            cx, cy, w, h = (float(r[k]) for k in ("cx", "cy", "w", "h"))
            ax.add_patch(plt.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, ec="red", lw=1.5))
        ax.set_title(f"{sp}, {len(by_img[(iid, sp)])} plate(s)")
    for ax in axes.flat:
        ax.axis("off")
plt.tight_layout()
plt.show()
print(f"elapsed {time.perf_counter() - t0:.1f} s")